# Robustness Tests: Alternative Outcome Measures

Six alternative dependent variables to confirm the main anti-labor intensity result
is not an artifact of the LLM sentiment classifier.

1. Keyword-based anti-labor score
2. Raw signed sentiment confidence
3. Refined keyword net score (anti vs. pro labor)
4. Outcome framing vocabulary
5. Source attribution (management vs. worker quotes)
6. TF-IDF vocabulary gap

In [67]:
import pandas as pd
import sqlite3
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col
import warnings
warnings.filterwarnings('ignore')

DB_PATH = '../data/newspapers.db'
OE_PATH = '../data/personnel_coding/owners_and_editors.csv'
CODE_PATH = '../data/personnel_coding/combined_coding.csv'

# Load intermediate data from 01_data_preparation
df = pd.read_csv('intermediate/analysis_sample.csv')
counts = pd.read_csv('intermediate/sentiment_counts.csv')
paper_year_rr = pd.read_csv('intermediate/paper_year_rr.csv')
person_rr = pd.read_csv('intermediate/person_rr.csv')

import numpy as np
import re
import hashlib

print(f'Loaded analysis sample: {len(df)} newspaper-years')

Loaded analysis sample: 265 newspaper-years


---

## 10. Test 1: Keyword-Based Anti-Labor Score

A model-free outcome: for each labor article (already classified by the sentiment pipeline) count whether it contains any of a set of anti-labor sentiment keywords (*mob, agitator, rioter, anarchist, communist*, etc.). The newspaper-year outcome is the **share of labor articles containing at least one anti-labor keyword**.

This test is independent of the Gemini classification model and validates whether the framing difference between railroad-tied and non-railroad papers is detectable in raw word choice.

In [68]:
import hashlib, json

ANTI_LABOR_KEYWORDS = [
    'mob', 'agitator', 'agitators', 'rioter', 'rioters',
    'anarchist', 'anarchists', 'communist', 'communists', 'communism',
    'lawless', 'incendiary', 'rabble', 'disorderly',
    'dangerous classes', 'tramp', 'vagabond', 'vagabonds',
    'instigator', 'instigators', 'troublemaker',
    'incited', 'inflamed', 'radical element',
]

# Filtered list for regression (≥50 total hits across corpus)
ANTI_LABOR_KEYWORDS_FILTERED = [
    'mob', 'agitator', 'agitators', 'rioter', 'rioters',
    'anarchist', 'communist', 'communists', 'communism',
    'lawless', 'incendiary', 'disorderly', 'tramp',
]

def has_antilabor_keyword(text: str) -> int:
    t = text.lower()
    return int(any(kw in t for kw in ANTI_LABOR_KEYWORDS_FILTERED))

# Load pre-classified labor articles from JSON (includes full text)
# Combine labor_only + both (articles that are labor AND railroad)
with open('../sentiment_analysis/data/classified_articles/labor_only.json') as f:
    labor_json = json.load(f)
with open('../sentiment_analysis/data/classified_articles/both.json') as f:
    both_json = json.load(f)

labor_arts = pd.DataFrame(labor_json + both_json).drop_duplicates(subset='article_id')

# Filter to analysis sample (issn, year) pairs
analysis_pairs = set(zip(df['issn'], df['year']))
labor_arts = labor_arts[
    labor_arts.apply(lambda r: (r['issn'], r['year']) in analysis_pairs, axis=1)
].copy()

print(f"Labor articles in analysis sample: {len(labor_arts)}")
print(f"Distinct (issn, year) groups: {labor_arts.groupby(['issn','year']).ngroups}")
print(f"Keywords used (≥50 hits): {len(ANTI_LABOR_KEYWORDS_FILTERED)}/{len(ANTI_LABOR_KEYWORDS)}")

Labor articles in analysis sample: 12579
Distinct (issn, year) groups: 265
Keywords used (≥50 hits): 13/24


In [69]:
# Compute keyword scores directly on labor_arts — no DB query needed
labor_arts['keyword_hit'] = labor_arts['text'].apply(has_antilabor_keyword)
scores_df = labor_arts[['article_id', 'keyword_hit']].copy()
print(f"Scored {len(scores_df)} labor articles")

Scored 12579 labor articles


In [70]:
# Aggregate to newspaper-year: share of labor articles with any anti-labor keyword
kw_agg = (
    labor_arts.groupby(['issn', 'year'])
    .agg(kw_hits=('keyword_hit', 'sum'), total_scored=('keyword_hit', 'count'))
    .reset_index()
)
kw_agg['keyword_score'] = kw_agg['kw_hits'] / kw_agg['total_scored']

df_kw     = df.merge(kw_agg, on=['issn', 'year'], how='inner')
df_kw_pol = df_kw.dropna(subset=['republican']).copy()

print(f"Keyword regression sample: {len(df_kw)} obs")
print()
print(df_kw.groupby('railroad_interest')[['keyword_score']].describe().T)

Keyword regression sample: 265 obs

railroad_interest             0           1
keyword_score count  151.000000  114.000000
              mean     0.141383    0.128315
              std      0.158142    0.127380
              min      0.000000    0.000000
              25%      0.000000    0.027904
              50%      0.094737    0.101020
              75%      0.188312    0.180114
              max      0.750000    0.714286


In [71]:
# Test 1 regressions: keyword score as outcome
k1 = smf.ols('keyword_score ~ railroad_interest', data=df_kw).fit(cov_type='HC3')
k2 = smf.ols('keyword_score ~ railroad_interest + C(year)', data=df_kw).fit(cov_type='HC3')
k4 = smf.ols('keyword_score ~ railroad_interest + C(year) + republican',
             data=df_kw_pol).fit(cov_type='HC3')

print('=== Test 1: Keyword-Based Anti-Labor Score ===')
print(summary_col(
    [k1, k2, k4],
    model_names=['(1) Bivariate', '(2) Year FE', '(3) Year FE + Party'],
    stars=True,
    info_dict={'N': lambda m: m.nobs, 'R²': lambda m: round(m.rsquared, 3)},
    regressor_order=['railroad_interest', 'republican', 'Intercept']
))


=== Test 1: Keyword-Based Anti-Labor Score ===

                  (1) Bivariate (2) Year FE (3) Year FE + Party
---------------------------------------------------------------
railroad_interest -0.0131       -0.0228     -0.0412*           
                  (0.0176)      (0.0147)    (0.0239)           
republican                                  0.0042             
                                            (0.0285)           
Intercept         0.1414***     0.1066**    0.2176             
                  (0.0129)      (0.0449)    (0.1543)           
C(year)[T.1871]                 0.0215      -0.1010            
                                (0.0656)    (0.1972)           
C(year)[T.1872]                 -0.0057     -0.0931            
                                (0.0597)    (0.1751)           
C(year)[T.1873]                 -0.0104     -0.1154            
                                (0.0504)    (0.1624)           
C(year)[T.1876]                 0.0424      0.0390      

In [72]:
# Decomposition: per-keyword hit rate and total hits by railroad_interest
rr_map = df.groupby("issn")["railroad_interest"].max()
labor_arts["rr"] = labor_arts["issn"].map(rr_map)

n_rr   = (labor_arts["rr"] == 1).sum()
n_ctrl = (labor_arts["rr"] == 0).sum()

rows = []
for kw in ANTI_LABOR_KEYWORDS:
    hits = labor_arts["text"].str.lower().str.contains(kw, regex=False)
    rr_hits   = hits[labor_arts["rr"] == 1].sum()
    ctrl_hits = hits[labor_arts["rr"] == 0].sum()
    rr_rate   = rr_hits / n_rr
    ctrl_rate = ctrl_hits / n_ctrl
    rows.append({
        "keyword": kw,
        "total_hits": rr_hits + ctrl_hits,
        "rr_hits": rr_hits,
        "ctrl_hits": ctrl_hits,
        "railroad_rate": rr_rate,
        "control_rate": ctrl_rate,
        "diff (rr - ctrl)": rr_rate - ctrl_rate,
    })

decomp = pd.DataFrame(rows).sort_values("diff (rr - ctrl)", ascending=False)
decomp[["railroad_rate", "control_rate", "diff (rr - ctrl)"]] = \
    decomp[["railroad_rate", "control_rate", "diff (rr - ctrl)"]].round(4)

print("=== Test 1 Decomposition: Per-Keyword Hit Rate ===")
print(f"Railroad articles: {n_rr}, Control articles: {n_ctrl}")
print()
print(decomp.to_string(index=False))

=== Test 1 Decomposition: Per-Keyword Hit Rate ===
Railroad articles: 8060, Control articles: 4519

          keyword  total_hits  rr_hits  ctrl_hits  railroad_rate  control_rate  diff (rr - ctrl)
         agitator         148      106         42         0.0132        0.0093            0.0039
        agitators          97       73         24         0.0091        0.0053            0.0037
       communists         118       83         35         0.0103        0.0077            0.0026
        communist         227      152         75         0.0189        0.0166            0.0023
        anarchist          72       52         20         0.0065        0.0044            0.0020
        communism         104       72         32         0.0089        0.0071            0.0019
           rioter         296      194        102         0.0241        0.0226            0.0015
       anarchists          48       33         15         0.0041        0.0033            0.0008
          rioters         2

---

## 11. Test 2: Raw Sentiment Score as Outcome

Instead of a binary anti- / pro-labor classification, use the model's **confidence score** as a continuous directional signal:

- `+confidence` if the article is classified **anti_labor**
- `-confidence` if the article is classified **pro_labor**
- `0` if **neutral**

Aggregated to the newspaper-year as the **mean signed score** across all labor articles. Positive values indicate net anti-labor framing; negative values indicate net pro-labor framing. This exploits variation in framing intensity that the binary outcome discards.

In [44]:
# Compute signed sentiment score from labor_confidence
conn = sqlite3.connect(DB_PATH)
sent_conf = pd.read_sql(
    """SELECT issn, year, labor_sentiment, labor_confidence
       FROM article_sentiment
       WHERE labor_sentiment IS NOT NULL AND issn != ''""",
    conn
)
conn.close()

def signed_score(row):
    if row['labor_sentiment'] == 'anti_labor':
        return row['labor_confidence']
    elif row['labor_sentiment'] == 'pro_labor':
        return -row['labor_confidence']
    return 0.0

sent_conf['signed_score'] = sent_conf.apply(signed_score, axis=1)

# Aggregate to newspaper-year: mean signed score
signed_agg = (
    sent_conf.groupby(['issn', 'year'])
    .agg(mean_sentiment=('signed_score', 'mean'),
         n_articles=('signed_score', 'count'))
    .reset_index()
)

df_raw     = df.merge(signed_agg, on=['issn', 'year'], how='inner')
df_raw_pol = df_raw.dropna(subset=['republican']).copy()

print(f"Raw sentiment regression sample: {len(df_raw)} obs")
print(f"Signed score range: [{sent_conf['signed_score'].min():.3f}, {sent_conf['signed_score'].max():.3f}]")
print()
print('Mean signed score by treatment group:')
print(df_raw.groupby('railroad_interest')['mean_sentiment'].describe().T)


Raw sentiment regression sample: 265 obs
Signed score range: [-0.929, 0.955]

Mean signed score by treatment group:
railroad_interest           0           1
count              151.000000  114.000000
mean                -0.098977   -0.024972
std                  0.353209    0.314434
min                 -0.918000   -0.923500
25%                 -0.318292   -0.168007
50%                 -0.010455    0.005090
75%                  0.102033    0.172525
max                  0.918000    0.742167


In [45]:
# Test 2 regressions: mean signed sentiment as outcome
r1 = smf.ols('mean_sentiment ~ railroad_interest', data=df_raw).fit(cov_type='HC3')
r2 = smf.ols('mean_sentiment ~ railroad_interest + C(year)', data=df_raw).fit(cov_type='HC3')
r4 = smf.ols('mean_sentiment ~ railroad_interest + C(year) + republican',
             data=df_raw_pol).fit(cov_type='HC3')

print('=== Test 2: Raw Sentiment Score (Mean Signed Confidence) ===')
print(summary_col(
    [r1, r2, r4],
    model_names=['(1) Bivariate', '(2) Year FE', '(3) Year FE + Party'],
    stars=True,
    info_dict={'N': lambda m: m.nobs, 'R²': lambda m: round(m.rsquared, 3)},
    regressor_order=['railroad_interest', 'republican', 'Intercept']
))


=== Test 2: Raw Sentiment Score (Mean Signed Confidence) ===

                  (1) Bivariate (2) Year FE (3) Year FE + Party
---------------------------------------------------------------
railroad_interest 0.0740*       0.0672*     0.0586             
                  (0.0413)      (0.0406)    (0.0564)           
republican                                  0.1395**           
                                            (0.0560)           
Intercept         -0.0990***    -0.2311**   -0.3014            
                  (0.0288)      (0.1179)    (0.3679)           
C(year)[T.1871]                 -0.0255     -0.1803            
                                (0.1517)    (0.4067)           
C(year)[T.1872]                 0.1604      0.1407             
                                (0.1500)    (0.4031)           
C(year)[T.1873]                 0.1412      0.1280             
                                (0.1334)    (0.3783)           
C(year)[T.1876]                 0.1143    

---

## 12. Test 3: Refined Keyword Net Score

Expands the keyword approach into separate anti-labor and pro-labor lists focused on **framing language**
(rather than inflammatory nouns), then computes a per-article net score aggregated to newspaper-year level.

* **Anti-labor keywords:** order, property, trespass, lawless, dangerous, incendiary, rioters, agitators,
  troops, militia, dispersed, arrested, suppressed, quelled, violence, destruction, disturbance
* **Pro-labor keywords:** wages, grievance, organize, union, rights, fair, peaceable, assembly, demands,
  concession, settlement, workers, laborers, employees, justice

Net score = (anti\_hits − pro\_hits) / (anti\_hits + pro\_hits), 0 when neither list fires.

In [46]:
import re

ANTI_LABOR_KW = re.compile(
    r"\b(order|property|trespass|lawless|dangerous|incendiary|rioters?|agitators?"
    r"|troops?|militia|disperse[sd]?|arrest(?:ed|s)?|suppress(?:ed|ion)?|quell(?:ed)?"
    r"|violence|destruction|disturbance)\b",
    re.IGNORECASE
)
PRO_LABOR_KW = re.compile(
    r"\b(wages?|grievance|organiz(?:e[sd]?|ation)|union|rights?|fair|peaceable|assembly"
    r"|demands?|concession|settlement|workers?|labou?rers?|employees?|justice)\b",
    re.IGNORECASE
)

def net_kw_score(text):
    if not isinstance(text, str):
        return None
    anti = len(ANTI_LABOR_KW.findall(text))
    pro  = len(PRO_LABOR_KW.findall(text))
    denom = anti + pro
    if denom == 0:
        return 0.0
    return (anti - pro) / denom

# Reuse labor_arts from Test 1 bulk fetch — already filtered to labor articles
scores = []
for row in labor_arts.itertuples(index=False):
    score = net_kw_score(row.text)
    if score is not None:
        scores.append({'issn': row.issn, 'year': row.year, 'net_kw': score})

scores_df = pd.DataFrame(scores)

net_kw_agg = (
    scores_df.groupby(["issn", "year"])["net_kw"]
    .mean().reset_index()
    .rename(columns={"net_kw": "net_kw_score"})
)
df_netkw = df.merge(net_kw_agg, on=["issn","year"], how="inner")
print(f"Net keyword sample: {len(df_netkw)} newspaper-years")
print(df_netkw.groupby("railroad_interest")["net_kw_score"].describe().round(3))

Net keyword sample: 265 newspaper-years
                   count   mean    std  min    25%    50%    75%    max
railroad_interest                                                      
0                  151.0 -0.491  0.311 -1.0 -0.684 -0.535 -0.369  1.000
1                  114.0 -0.542  0.253 -1.0 -0.692 -0.552 -0.379  0.187


In [47]:
nk1 = smf.ols("net_kw_score ~ railroad_interest", data=df_netkw).fit(cov_type="HC3")
nk2 = smf.ols("net_kw_score ~ railroad_interest + C(year_str)", data=df_netkw).fit(cov_type="HC3")
nk3 = smf.ols("net_kw_score ~ railroad_interest + C(year_str) + C(state)", data=df_netkw).fit(cov_type="HC3")

print(summary_col(
    [nk1, nk2, nk3],
    model_names=["Bivariate","Year FE","Year+State FE"],
    stars=True,
    info_dict={"N": lambda m: f"{int(m.nobs)}", "R2": lambda m: f"{m.rsquared:.3f}"}
))


                                 Bivariate   Year FE   Year+State FE
--------------------------------------------------------------------
Intercept                        -0.4912*** -0.6138*** -0.3849***   
                                 (0.0254)   (0.0980)   (0.1433)     
railroad_interest                -0.0504    -0.0661**  -0.2716***   
                                 (0.0348)   (0.0315)   (0.0995)     
C(year_str)[T.1871]                         0.1697     0.3180*      
                                            (0.1291)   (0.1772)     
C(year_str)[T.1872]                         0.0706     0.2415**     
                                            (0.1199)   (0.1177)     
C(year_str)[T.1873]                         -0.0207    0.1117       
                                            (0.1187)   (0.1158)     
C(year_str)[T.1876]                         0.3420**   0.4819***    
                                            (0.1334)   (0.1290)     
C(year_str)[T.1877]              

---

## 13. Test 4: Outcome Framing Vocabulary

Focuses on how papers describe *the resolution* of labor disputes — the most unambiguous framing signal.

* **Anti-labor outcome words:** dispersed, arrested, suppressed, defeated, quelled, crushed, routed, driven off, expelled
* **Pro-labor outcome words:** settled, conceded, won, recognized, granted, agreed, succeeded, secured, achieved

Score = (anti\_outcome\_hits − pro\_outcome\_hits) / (anti + pro), excluding articles with no outcome vocabulary.

In [48]:
ANTI_OUTCOME = re.compile(
    r"\b(disperse[sd]?|arrest(?:ed|s)?|suppress(?:ed)?|defeat(?:ed)?|quell(?:ed)?"
    r"|crush(?:ed)?|rout(?:ed)?|driven\s+off|expell?(?:ed)?)\b",
    re.IGNORECASE
)
PRO_OUTCOME = re.compile(
    r"\b(settle[sd]?|settlement|conced(?:ed)?|won|recogniz(?:ed)?|grant(?:ed)?"
    r"|agre(?:ed)?|succeed(?:ed)?|secur(?:ed)?|achiev(?:ed)?)\b",
    re.IGNORECASE
)

def outcome_score(text):
    if not isinstance(text, str):
        return None
    anti = len(ANTI_OUTCOME.findall(text))
    pro  = len(PRO_OUTCOME.findall(text))
    denom = anti + pro
    if denom == 0:
        return 0.0
    return (anti - pro) / denom

# Reuse labor_arts from Test 1 bulk fetch — already filtered to labor articles
outcome_scores = []
for row in labor_arts.itertuples(index=False):
    score = outcome_score(row.text)
    if score is not None:
        outcome_scores.append({'issn': row.issn, 'year': row.year, 'outcome_framing': score})

outcome_df = pd.DataFrame(outcome_scores)

outcome_agg = (
    outcome_df.groupby(["issn","year"])["outcome_framing"]
    .mean().reset_index()
    .rename(columns={"outcome_framing": "outcome_score"})
)
df_outcome = df.merge(outcome_agg, on=["issn","year"], how="inner")
print(f"Outcome framing sample: {len(df_outcome)} newspaper-years")
print(df_outcome.groupby("railroad_interest")["outcome_score"].describe().round(3))

Outcome framing sample: 265 newspaper-years
                   count   mean    std  min    25%    50%  75%  max
railroad_interest                                                  
0                  151.0 -0.158  0.282 -1.0 -0.286 -0.163  0.0  1.0
1                  114.0 -0.138  0.219 -1.0 -0.249 -0.121  0.0  1.0


In [49]:
oc1 = smf.ols("outcome_score ~ railroad_interest", data=df_outcome).fit(cov_type="HC3")
oc2 = smf.ols("outcome_score ~ railroad_interest + C(year_str)", data=df_outcome).fit(cov_type="HC3")
oc3 = smf.ols("outcome_score ~ railroad_interest + C(year_str) + C(state)", data=df_outcome).fit(cov_type="HC3")

print(summary_col(
    [oc1, oc2, oc3],
    model_names=["Bivariate","Year FE","Year+State FE"],
    stars=True,
    info_dict={"N": lambda m: f"{int(m.nobs)}", "R2": lambda m: f"{m.rsquared:.3f}"}
))


                                 Bivariate  Year FE  Year+State FE
------------------------------------------------------------------
Intercept                        -0.1580*** -0.1812  -0.1782      
                                 (0.0231)   (0.1398) (0.1475)     
railroad_interest                0.0196     0.0080   0.0376       
                                 (0.0309)   (0.0312) (0.0864)     
C(year_str)[T.1871]                         -0.1108  -0.0551      
                                            (0.1582) (0.1544)     
C(year_str)[T.1872]                         -0.0568  0.0161       
                                            (0.1435) (0.1341)     
C(year_str)[T.1873]                         0.0345   0.1083       
                                            (0.1501) (0.1344)     
C(year_str)[T.1876]                         0.1911   0.2988       
                                            (0.1944) (0.2131)     
C(year_str)[T.1877]                         0.2056   0.2845**

In [50]:
# labor_arts already has text for all labor articles — just alias it
labor_articles = labor_arts.copy()
print(f'Built labor_articles: {len(labor_articles)} rows')

Built labor_articles: 12579 rows


---

## 15. Test 5: TF-IDF Vocabulary Gap

Unsupervised check: fit TF-IDF on railroad-tied vs. non-railroad paper corpora and compare term weights.
Surfaces framing differences not anticipated by the researcher and can inform future keyword lists.

Includes unigrams and bigrams; standard English stopwords removed.

In [54]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Map issn -> railroad_interest (take max to resolve duplicates — 1 if any entry is railroad-tied)
rr_map = df.groupby("issn")["railroad_interest"].max()
labor_articles["rr"] = labor_articles["issn"].map(rr_map)

rr_corpus   = " ".join(labor_articles[labor_articles["rr"] == 1]["text"].dropna().tolist())
ctrl_corpus = " ".join(labor_articles[labor_articles["rr"] == 0]["text"].dropna().tolist())

print(f"Railroad corpus:  {len(rr_corpus):>12,} chars")
print(f"Control corpus:   {len(ctrl_corpus):>12,} chars")

# OCR artifact stop list — common mis-scanned fragments in 19th-century newspaper digitizations
OCR_STOPWORDS = {
    'tho', 'thu', 'tbe', 'tlie', 'tiie', 'tne', 'thc',
    'ana', 'aud', 'anu', 'anil', 'ami', 'arid',
    'tins', 'vino', 'arc',
    'oi', 'ot', 'ol', 'oc', 'od', 'ob',
    'ii', 'iii', 'iv', 'vi', 'vii', 'viii', 'ix', 'xi', 'xii',
    'tor', 'ter', 'tion', 'tions', 'ing', 'ings', 'ment', 'ness',
    'tue', 'tuo', 'tbe', 'hie', 'hia', 'bis', 'bia',
    'tie', 'tic', 'tin', 'lio', 'lin',
    'ne', 'na', 'ni', 'nu', 'ni',
    'oh', 'ot', 'op', 'oe', 'od',
    'lie', 'lias', 'las', 'liad', 'lar',
    'nave', 'bave', 'liave', 'lave',
    'bo', 'ba', 'be', 'bi', 'bu',
    'wa', 'waa', 'wai', 'wns', 'wns',
    'ho', 'ha', 'hi', 'hu', 'hc',
    'io', 'ia', 'ie', 'iu', 'ir',
    'ro', 'ra', 're', 'ri', 'ru',
    'de', 'da', 'di', 'du', 'dv',
    'se', 'si', 'su', 'sl', 'sr',
    'cr', 'ct', 'cl', 'co', 'ca',
    'mt', 'ft', 'st', 'lt', 'ct',
    'ist', 'ism', 'ity', 'ies', 'ily',
    'ed', 'er', 'al', 'ly', 'en', 'le', 'el',
    '10', '11', '12', '15', '20', '25', '30', '40', '50', '60', '70', '75', '80', '90', '100',
}

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
combined_stops = list(ENGLISH_STOP_WORDS | OCR_STOPWORDS)

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words=combined_stops,
    token_pattern=r"\b[a-zA-Z]{3,}\b",  # min 3 chars, alpha only — filters stray digits/fragments
    min_df=1
)
mat   = tfidf.fit_transform([rr_corpus, ctrl_corpus])
terms = np.array(tfidf.get_feature_names_out())

scores_rr   = mat[0].toarray().flatten()
scores_ctrl = mat[1].toarray().flatten()
diff = scores_rr - scores_ctrl

top_rr_idx   = np.argsort(-diff)[:30]
top_ctrl_idx = np.argsort( diff)[:30]

print("\n=== Top 30 terms OVER-INDEXING in railroad-tied papers ===")
for t, s in zip(terms[top_rr_idx], diff[top_rr_idx]):
    print(f"  {t:<35} +{s:.4f}")

print("\n=== Top 30 terms OVER-INDEXING in non-railroad (control) papers ===")
for t, s in zip(terms[top_ctrl_idx], -diff[top_ctrl_idx]):
    print(f"  {t:<35} +{s:.4f}")

Railroad corpus:    24,725,543 chars
Control corpus:     13,983,133 chars

=== Top 30 terms OVER-INDEXING in railroad-tied papers ===
  strike                              +0.0452
  men                                 +0.0426
  work                                +0.0417
  union                               +0.0298
  strikers                            +0.0289
  meeting                             +0.0208
  company                             +0.0191
  central                             +0.0152
  association                         +0.0148
  manufacturers                       +0.0145
  coal                                +0.0130
  oil                                 +0.0121
  held                                +0.0119
  city                                +0.0106
  day                                 +0.0105
  members                             +0.0103
  movement                            +0.0099
  miners                              +0.0099
  trouble                             